# Train a CARE model (demo-first, reusable)

This notebook trains a CARE denoising model from paired training patches (`.npz`) created in the **datagen** notebook.

The workflow:
1. Load the patch dataset `(X, Y)`
2. Configure the CARE model
3. Train
4. Plot training curves
5. Run predictions on validation patches

Edit only the **CONFIG** cell to switch between the demo dataset and your own dataset.

In [ ]:
# --- Notebook setup (CARE env) ---

import os
import subprocess

from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

%matplotlib inline
%config InlineBackend.figure_format = "retina"

from csbdeep.utils import axes_dict, plot_some, plot_history
from csbdeep.io import load_training_data
from csbdeep.models import Config, CARE

# Reduce TensorFlow log noise
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

### TensorFlow / GPU initialization

On shared machines, enabling **memory growth** prevents TensorFlow from reserving all GPU memory at startup.

In [ ]:
def get_free_gpu():
    result = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=memory.used", "--format=csv,nounits,noheader"]
    )
    memory_used = [int(x) for x in result.decode("utf-8").strip().split("\n")]
    
    gpu_id = memory_used.index(min(memory_used))
    return gpu_id

gpu_id = get_free_gpu()

os.environ["CUDA_VISIBLE_DEVICES"] = str(gpu_id)

print(f"Using GPU {gpu_id}")

In [ ]:
import tensorflow as tf

gpus = tf.config.list_physical_devices("GPU")
for gpu in gpus:
    tf.config.experimental.set_memory_growth(gpu, True)

print("TensorFlow:", tf.__version__)
print("Visible GPUs:", len(gpus))

### CONFIG

- If `USE_EXAMPLE_DATA=True`, this notebook uses `CARE_example_data/` next to the notebook.
- If `False`, set your custom dataset path and (optionally) choose a patch file.

Expected output from the datagen notebook:
- `DATA_ROOT/train_patches/*.npz`

In [ ]:
# ============================================================
# CONFIG (edit this cell for a new dataset)
# ============================================================

USE_EXAMPLE_DATA = True

# Demo dataset root (relative to notebook)
DEMO_DATA_ROOT = Path("CARE_example_data")

# Custom dataset root (absolute or relative)
CUSTOM_DATA_ROOT = Path("/path/to/your/dataset")   # <-- change this

# Choose which root to use
DATA_ROOT = DEMO_DATA_ROOT if USE_EXAMPLE_DATA else CUSTOM_DATA_ROOT
DATA_ROOT = DATA_ROOT.expanduser().resolve()

# Patch folder naming (supports a legacy folder called "train patches")
PATCH_DIR_CANDIDATES = ["train_patches", "train patches"]

# Optional: specify a particular patch file name (otherwise first .npz is used)
PATCH_FILE_NAME = None   # e.g. "my_train_patches.npz"

# Model output
MODEL_DIR = (DATA_ROOT / "models").resolve()
MODEL_DIR.mkdir(parents=True, exist_ok=True)

model_name = "care_denoise_demo" if USE_EXAMPLE_DATA else "care_model"

print("DATA_ROOT:", DATA_ROOT)
print("MODEL_DIR:", MODEL_DIR)
print("model_name:", model_name)

### Locate training patches

The patch dataset (`.npz`) generated in the **datagen notebook** is stored inside:

```
DATA_ROOT/train_patches/
```

This cell automatically finds the patch directory and loads the first `.npz` file it contains.

To load a specific patch file instead, set in the **CONFIG cell**:

```
PATCH_FILE_NAME = "my_training_patches.npz"
```

In [ ]:
def find_patch_dir(data_root: Path, candidates=("train_patches", "train patches")) -> Path:
    for name in candidates:
        p = data_root / name
        if p.exists():
            return p
    # fall back to preferred name
    return data_root / candidates[0]

PATCH_DIR = find_patch_dir(DATA_ROOT, PATCH_DIR_CANDIDATES)
PATCH_DIR.mkdir(parents=True, exist_ok=True)

if PATCH_FILE_NAME is None:
    patch_files = sorted(PATCH_DIR.glob("*.npz"))
    assert patch_files, f"No .npz patch files found in {PATCH_DIR}"
    PATCH_FILE = patch_files[0]
else:
    PATCH_FILE = PATCH_DIR / PATCH_FILE_NAME
    assert PATCH_FILE.exists(), f"Patch file not found: {PATCH_FILE}"

print("PATCH_DIR:", PATCH_DIR.resolve())
print("PATCH_FILE:", PATCH_FILE.resolve())

### Load training patches

The patch dataset contains:

- `X` : input patches (model input)  
- `Y` : target patches (ground truth)

A fraction of the patches is automatically held out as a **validation set**, which is used during training to monitor model performance and detect overfitting.

The validation fraction is controlled by the parameter:

```
validation_split
```

For example:

```
validation_split = 0.05
```

means **5% of the patches are used for validation** and the remaining **95% for training**.

Typical values:

- `0.05` (default, good for most datasets)
- `0.1` if you want a larger validation set
- smaller values if your dataset is very small

In [ ]:
(X, Y), (X_val, Y_val), axes = load_training_data(
    str(PATCH_FILE),
    validation_split=0.05,
)

c = axes_dict(axes)["C"]
n_channel_in  = X.shape[c]
n_channel_out = Y.shape[c]

print("Axes:", axes)
print("X:", X.shape, "Y:", Y.shape)
print("X_val:", X_val.shape, "Y_val:", Y_val.shape)
print("Channels in/out:", n_channel_in, "/", n_channel_out)

### Sanity check (visual)

Top row: input patches (`X`)  
Bottom row: target patches (`Y`)

In [ ]:
n_show = 5
plt.figure(figsize=(12, 5))
plot_some(X_val[:n_show], Y_val[:n_show])
plt.suptitle("Validation patch pairs (top: input, bottom: target)")
plt.show()

### CARE model configuration

Before constructing the CARE model, we define its training configuration using a `Config` object.

This configuration specifies key aspects of the network and training procedure, including:

- the architecture of the underlying **U-Net**
- the **batch size**
- the number of **training steps per epoch**
- the number of **training epochs**
- whether the model is **probabilistic**

In this notebook the configuration can be used both for **demo runs** and **full training experiments**.

For demo runs we use smaller values so the notebook executes quickly.  
For real datasets these values are typically increased.

For example:

```
train_steps_per_epoch = 100
train_epochs = 10
```

Typical values for real training:

```
train_steps_per_epoch = 200–400
train_epochs = 50–200
```

The default batch size of `8` works on most GPUs.  
If you encounter GPU memory issues, reduce `train_batch_size` or decrease the patch size in the data generation notebook.

In [ ]:
config = Config(
    axes=axes,
    n_channel_in=n_channel_in,
    n_channel_out=n_channel_out,
    unet_kern_size=3,
    train_batch_size=8,
    train_steps_per_epoch=100,  # demo-friendly
    train_epochs=10,            # demo-friendly
)

print(config)

### Create the CARE model

The model is saved automatically in:

`MODEL_DIR / model_name /`

This folder will contain checkpoints, logs, and the best weights.

In [ ]:
model = CARE(
    config=config,
    name=model_name,
    basedir=str(MODEL_DIR),
)

print("Model folder:", (MODEL_DIR / model_name).resolve())

### Inspect the model architecture

We can inspect the underlying Keras model to verify:

- input shape
- number of feature maps per level
- depth of the U-Net
- total number of trainable parameters

The architecture is automatically constructed from the `Config` object.



In [ ]:
model.keras_model._name = model_name
model.keras_model.summary()

### Train

This trains the model on `(X, Y)` and monitors performance on `(X_val, Y_val)`.

In [ ]:
history = model.train(
    X, Y,
    validation_data=(X_val, Y_val),
)

print("Training finished.")

### Training curves

We plot loss/metrics from training and validation over time.

In [ ]:
print("Available metrics:", sorted(history.history.keys()))

plt.figure(figsize=(12, 4))
plot_history(
    history,
    ['loss', 'val_loss'],
    ['mse', 'val_mse', 'mae', 'val_mae'],
)
plt.tight_layout()
plt.show()

### Evaluate on validation patches

We predict on a small batch of validation patches and compare:

- input
- target
- prediction

In [ ]:
n_examples = 5

X_example = X_val[:n_examples]
Y_example = Y_val[:n_examples]

Y_pred = model.keras_model.predict(X_example, verbose=0)

# If probabilistic model, keep only mean prediction
if config.probabilistic:
    Y_pred = Y_pred[..., : Y_pred.shape[-1] // 2]

plt.figure(figsize=(20, 12))
plot_some(X_example, Y_example, Y_pred, pmax=99.5)
plt.suptitle(
    f"{n_examples} validation patches\n"
    "top: input, middle: target, bottom: prediction",
    y=0.98,
)
plt.show()

## Load the model later (Python)

CARE saves the model to:

`MODEL_DIR / model_name /`

To load it in another notebook:

```python
from csbdeep.models import CARE
model = CARE(config=None, name=model_name, basedir=str(MODEL_DIR))